# Rebuilding the FDA's count of novel drugs for 2023 from Drugs@FDA

**Data kit lab** `fda-novel-approvals-2023` · Regulatory · about 40 minutes

**The question.** How many novel drugs did the FDA's drug centre (CDER) approve in calendar year 2023, and can that count be rebuilt from the agency's own application records in Drugs@FDA?

**What you do.** You open the Drugs@FDA bulk file from openFDA, turn it into one row per submission, and apply the agency's own definition of a novel drug as far as the fields allow. Then you compare your count with the number the FDA prints on its page *Novel Drug Approvals for 2023*, and you match the two lists drug by drug.

**The rule used here was fixed before the data were read** (see `SPEC.txt` in the lab folder). It is not adjusted to match the published number. If the count differs, the lab says so and shows why.

**Sources.** Data: openFDA, Drugs@FDA (CC0 1.0 unless otherwise noted, https://open.fda.gov/license/). Published count: https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-2023 . Definition of the list: https://www.fda.gov/drugs/development-approval-process-drugs/novel-drug-approvals-fda .

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'fda-novel-approvals-2023'
FILES = json.loads(r'''{
 "drugsfda": {
  "url": "https://download.open.fda.gov/drug/drugsfda/drug-drugsfda-0001-of-0001.json.zip",
  "name": "drugsfda.zip",
  "sha256": "8ec95c8ed4c728d0fae13b089f5ba8f75f9d1548b5b3db2f2ef588fe00ced8fc",
  "bytes": 9431497
 },
 "novel2023": {
  "url": "https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-2023",
  "name": "novel2023",
  "sha256": "e9f478f264c9afa3ca5e1b1d0f2ca5f17efaf8001f7193897ada31b5df94e377",
  "bytes": 66527
 }
}''')
PUBLISHED = json.loads(r'''{
 "novel_approvals_2023": {
  "label": "Novel drugs approved by CDER in 2023",
  "value": 55,
  "tolerance": {
   "exact": true
  }
 }
}''')
PUBLISHED_FROM = 'FDA web page Novel Drug Approvals for 2023 (no page numbers): section FDA Novel Drug Therapy Approvals for 2023, the sentence giving the count; checked against the numbered list | https://www.fda.gov/drugs/novel-drug-approvals-fda/novel-drug-approvals-2023'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Open the bulk file

The zip holds one JSON file with a `meta` block and a list of applications in `results`. Each application has an `application_number`, a sponsor, its products and its submissions.

The first cell above checks each download against the copy this lab was tested with. openFDA re-exports this file often, so a notice that `drugsfda` differs is expected. Read `last_updated` below and note it with your count.

In [ ]:
import json
import re
import zipfile

import pandas as pd

with zipfile.ZipFile(data_file("drugsfda")) as z:
    print("files in the zip:", z.namelist())
    doc = json.loads(z.read(z.namelist()[0]))

meta = doc["meta"]
apps = doc["results"]
print("data last updated:", meta.get("last_updated"))
print("applications in the file:", len(apps), "| meta says:", meta.get("results", {}).get("total"))
print("keys of one application:", sorted(apps[0].keys()))

## 2. Application types

The prefix of `application_number` gives the type: NDA (new drug application), ANDA (abbreviated application, for generics) or BLA (biologics licence application). A generic is never novel, so this lab keeps NDA and BLA only.

In [ ]:
prefix = pd.Series([re.match(r"[A-Z]+", a["application_number"]).group(0) for a in apps]).value_counts()
print(prefix.to_string())
keep_apps = [a for a in apps if a["application_number"].startswith(("NDA", "BLA"))]
print("NDA and BLA applications kept:", len(keep_apps))

## 3. One row per submission

An application collects many submissions over its life: the original (`ORIG`) and later supplements (`SUPPL`). Each has its own status and status date. A new drug is approved through its original submission, so the filter is: type `ORIG`, status `AP` (approved), status date in 2023.

The field list describes `submission_status_date` as the date the status was applied to the submission.

In [ ]:
rows = []
for a in keep_apps:
    for s in a.get("submissions", []):
        rows.append({
            "application_number": a["application_number"],
            "appl_type": a["application_number"][:3],
            "sponsor_name": a.get("sponsor_name", ""),
            "brand_names": "; ".join(sorted({p.get("brand_name", "") for p in a.get("products", [])})),
            "submission_type": s.get("submission_type"),
            "submission_number": s.get("submission_number"),
            "submission_status": s.get("submission_status"),
            "submission_status_date": s.get("submission_status_date", ""),
            "submission_class_code": s.get("submission_class_code"),
        })
subs = pd.DataFrame(rows)
print("submissions of NDA and BLA applications:", len(subs))

in_2023 = subs["submission_status_date"].str.slice(0, 4) == "2023"
orig_2023 = subs[(subs["submission_type"] == "ORIG") & (subs["submission_status"] == "AP") & in_2023].copy()
print("original submissions approved in 2023:", len(orig_2023))
print(orig_2023.groupby("appl_type").size().to_string())
print("submission numbers seen:", orig_2023["submission_number"].value_counts().to_dict())

## 4. Filter submissions, not applications

A filter written on whole applications asks two separate questions: does the application have an original submission, and does it have any approval dated 2023? An old application with a 2023 supplement answers yes to both. The cell counts both ways.

In [ ]:
g = subs.assign(is_orig=subs["submission_type"] == "ORIG",
                ap_2023=(subs["submission_status"] == "AP") & in_2023).groupby(["application_number", "appl_type"])
loose = g.agg(has_orig=("is_orig", "any"), has_ap_2023=("ap_2023", "any")).reset_index()
loose = loose[loose["has_orig"] & loose["has_ap_2023"]]
print("whole-application filter:", loose.groupby("appl_type").size().to_dict())
print("single-submission filter:", orig_2023.groupby("appl_type")["application_number"].nunique().to_dict())

## 5. NDAs: new molecular entities

The FDA's list holds CDER's *new molecular entities* and *new therapeutic biological products*. For an NDA, the review class in `submission_class_code` carries the first idea: the Drugs@FDA glossary says Chemical Type 1 is given to an active ingredient never marketed in the US in any form. A code can join two types (for example Type 1 with Type 4, a new combination), so the cell reads the numbers in the code and keeps the NDA if 1 is among them. Type 10 is not Type 1.

In [ ]:
def type_numbers(code):
    if not isinstance(code, str):
        return set()
    return {int(n) for n in re.findall(r"\d+", code)}

nda = orig_2023[orig_2023["appl_type"] == "NDA"].copy()
print("class codes among 2023 original NDA approvals:")
print(nda["submission_class_code"].value_counts(dropna=False).to_string())
nda_novel = nda[nda["submission_class_code"].map(lambda c: 1 in type_numbers(c))]
print("NDAs with Type 1 in the class code:", nda_novel["application_number"].nunique())

## 6. BLAs: new therapeutic biological products

No documented Drugs@FDA field says whether a BLA is a new therapeutic biologic or a biosimilar of one already approved. The rule fixed for this lab therefore counts every original BLA approved in 2023. The class codes seen on these BLAs are printed for you to look at; they are not used, because no opened source defines them for BLAs.

In [ ]:
bla = orig_2023[orig_2023["appl_type"] == "BLA"].copy()
print("original BLA approvals in 2023:", bla["application_number"].nunique())
print("class codes seen on them (shown only):", bla["submission_class_code"].value_counts(dropna=False).to_dict())

## 7. The rebuilt count

In [ ]:
novel_apps = pd.concat([nda_novel, bla]).drop_duplicates("application_number").sort_values("submission_status_date")
rebuilt = int(novel_apps["application_number"].nunique())
print("NDA part:", nda_novel["application_number"].nunique(), "| BLA part:", bla["application_number"].nunique())
print("rebuilt count of novel approvals in 2023:", rebuilt)

## 8. Match the two lists drug by drug

The FDA page lists each novel drug with a link to its label. The label's file name starts with the application number (six digits). The cell reads the page as published, takes that number from each row, and sorts the drugs into three groups. Nothing from the page is typed here.

In [ ]:
import html as _html

page = data_file("novel2023").read_text(encoding="utf-8")

def plain(fragment):
    return " ".join(_html.unescape(re.sub(r"<[^>]+>", " ", fragment)).split())

listed = []
for tr in re.findall(r"<tr[ >].*?</tr>", page, re.S):
    cells = re.findall(r"<t[dh][ >].*?</t[dh]>", tr, re.S)
    if not cells or not re.fullmatch(r"\d+\.", plain(cells[0])):
        continue
    links = re.findall(r"/label/\d{4}/(\d{6})", tr)
    listed.append({"no": plain(cells[0]), "drug": plain(cells[1]), "ingredient": plain(cells[2]),
                   "date": plain(cells[3]), "appl_no": links[0] if links else None})
listed = pd.DataFrame(listed)
print("rows in the published list:", len(listed), "| rows without a label link:", int(listed["appl_no"].isna().sum()))

digits = {n[-6:]: n for n in subs["application_number"].unique()}
listed["application_number"] = listed["appl_no"].map(digits)
found = set(novel_apps["application_number"])
listed["found_by_rule"] = listed["application_number"].isin(found)

print("\n(a) listed and found by the rule:", int(listed["found_by_rule"].sum()))
missed = listed[~listed["found_by_rule"]].merge(
    subs[subs["submission_type"] == "ORIG"][["application_number", "submission_status", "submission_status_date", "submission_class_code"]],
    on="application_number", how="left")
print("(b) listed but not found by the rule:", len(listed) - int(listed["found_by_rule"].sum()))
print(missed[["no", "drug", "appl_no", "application_number", "submission_status", "submission_status_date", "submission_class_code"]].to_string(index=False))
extra = novel_apps[~novel_apps["application_number"].isin(set(listed["application_number"].dropna()))]
print("\n(c) counted by the rule but not on the list:", len(extra))
print(extra[["application_number", "brand_names", "sponsor_name", "submission_status_date", "submission_class_code"]].to_string(index=False))

## 9. Report to the harness

The harness compares the rebuilt count with the number the page prints (copied by `extract_expected.py`; the match must be exact). Read the result together with section 8: a difference has names behind it.

In [ ]:
RESULTS = {"novel_approvals_2023": rebuilt}
print(RESULTS)
report(RESULTS)

## What to take away

- "Novel" is the agency's curated label. For NDAs, the review class Type 1 carries most of it. For BLAs, the records alone do not separate a new biologic from a biosimilar, so a count from the records needs a second source (for example a biosimilar list) or a reviewed exception list. This lab leaves that out on purpose and shows the effect.
- Filter single submissions. A filter on applications lets in old products with new supplements.
- Write down the date of the data and the date of the page with every count. Both change.

## Exercises

1. Look at group (c). Look up one of those applications in Drugs@FDA (the FDA page links to it) and decide whether it is a biosimilar, a new biologic, or something else.
2. Look at group (b), if it is not empty. Is the miss caused by the date, the class code, the submission type, or the application type?
3. Change the date filter to the approval year of a different list page (for example 2022) and compare with that page. Do not change the rule.
4. Count the original approvals of 2023 by `submission_class_code` across all NDAs. Which types are most common, and what do they mean?
5. Apply the Type 1 test to the BLAs as well and count again. Then ask: which opened source says that the review class means the same thing on a BLA as on an NDA? A rule that matches after the fact is a hypothesis until a source supports it.
